In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"mahmoudahmedmohran","key":"c5fbb9c74d87cb5a4bd696a294a9b41c"}'}

In [9]:
pip install kaggle

In [10]:
# إنشاء مجلد مخفي باسم kaggle
!mkdir -p ~/.kaggle

# نقل الملف الذي رفعته إلى المجلد الجديد
!cp kaggle.json ~/.kaggle/

# تأمين الملف (إعطاء صلاحية القراءة لصاحب الملف فقط)
!chmod 600 ~/.kaggle/kaggle.json

In [11]:
import os

# تحميل ملفات المسابقة
# Attempt to download the competition data
!kaggle competitions download -c terrain-prices-reggression

# Define the expected zip file name (Kaggle usually names it after the competition)
zip_file_name = 'terrain-prices-reggression.zip'

# فك الضغط عن الملفات المحملة
# Check if the zip file was downloaded successfully before attempting to unzip
if os.path.exists(zip_file_name):
    print(f"'{zip_file_name}' found. Attempting to unzip...")
    !unzip -o {zip_file_name} # -o option to overwrite files without prompting
    print(f"Successfully unzipped '{zip_file_name}'.")
else:
    print(f"'{zip_file_name}' not found. Download might have failed or the file name is incorrect.")
    print("Please ensure your Kaggle API key is correct and you have accepted the competition rules on Kaggle. The download command previously resulted in '401 Client Error: Unauthorized'.")


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
'terrain-prices-reggression.zip' not found. Download might have failed or the file name is incorrect.
Please ensure your Kaggle API key is correct and you have accepted the competition rules on Kaggle. The download command previously resulted in '401 Client Error: Unauthorized'.


### Save Data to Google Drive

First, I will create a folder in your Google Drive to store the data, and then move the downloaded CSV files into this folder.

In [8]:
# Define the target directory in Google Drive
drive_folder = '/content/drive/MyDrive/Terrain Prices Reggression'

# Create the directory if it doesn't exist
!mkdir -p "{drive_folder}"

# Move the CSV files to the Google Drive folder
!mv train.csv "{drive_folder}/train.csv"
!mv test.csv "{drive_folder}/test.csv"
!mv sample_submission.csv "{drive_folder}/sample_submission.csv"

print(f"Files moved to: {drive_folder}")
!ls "{drive_folder}"

mv: cannot stat 'train.csv': No such file or directory
mv: cannot stat 'test.csv': No such file or directory
mv: cannot stat 'sample_submission.csv': No such file or directory
Files moved to: /content/drive/MyDrive/Terrain Prices Reggression


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to the train.csv file in Google Drive
train_csv_path = f"{drive_folder}/train.csv"

# Read the train.csv file into a pandas DataFrame
df = pd.read_csv(train_csv_path)



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Terrain Prices Reggression/train.csv'

## Read Data

In [ ]:
# Display the first 5 rows of the DataFrame
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(3)

##EDA (Exploratory Data Analysis)


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe().T

In [ ]:
df.describe(include= "object")

In [ ]:
df['zoning_code'].unique()

In [ ]:
df['land_use'].unique()

In [ ]:
df['location_type'].unique()

In [ ]:
df.describe(include= "int64").T

In [ ]:
df.describe(include= "float64").T

In [ ]:
df.shape


In [ ]:
df.columns

##Data Cleaning

In [ ]:
#check missing values
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

##Outliers

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns

outliers_data = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if not outliers.empty:
        outliers_data[col] = outliers[[col]].copy()
        print(f"--- Outliers found in column: {col} ---")
        print(f"Number of outliers: {len(outliers)}")
        print(outliers[[col]].head())
        print("\n")

if not outliers_data:
    print("No significant outliers detected using the IQR method in numerical columns.")
else:
    print(f"Summary: Outliers found in {len(outliers_data)} numerical columns.")

## Visualize Outliers with Box Plots

To better understand the distribution of outliers, we can visualize them using box plots for the numerical columns identified in the previous step.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the list of columns with outliers from the 'outliers_data' dictionary
columns_with_outliers = list(outliers_data.keys())

if columns_with_outliers:
    print(f"Generating box plots for {len(columns_with_outliers)} columns with detected outliers.")

    # Determine the number of rows and columns for the subplots
    n_cols = 3  # You can adjust this number
    n_rows = (len(columns_with_outliers) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten() # Flatten the array of axes for easy iteration

    for i, col in enumerate(columns_with_outliers):
        sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
        axes[i].set_title(f'Box Plot of {col}')
        axes[i].set_ylabel('') # Hide y-label to avoid clutter

    # Hide any unused subplots
    for j in range(len(columns_with_outliers), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
else:
    print("No outliers were detected in numerical columns, so no box plots will be generated.")

### Handling Outliers using Capping (IQR Method)

I will apply capping to the identified numerical columns using the IQR method. Values below the lower bound will be replaced by the lower bound, and values above the upper bound will be replaced by the upper bound. This helps to reduce the impact of extreme values while retaining the overall data distribution.

In [ ]:
for col in columns_with_outliers:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Apply capping directly to df
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print("Outliers have been capped for the identified numerical columns directly in the DataFrame 'df'.")
# Display descriptive statistics after capping to see the effect
df[columns_with_outliers].describe()

## Re-checking Outliers in `df_cleaned` after Capping

Now, let's verify if there are any remaining outliers in `df_cleaned` and visualize them.

In [ ]:
numerical_cols_cleaned = df.select_dtypes(include=np.number).columns
outliers_data_cleaned = {}

print("Re-detecting outliers in df after capping...")
for col in numerical_cols_cleaned:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if not outliers.empty:
        outliers_data_cleaned[col] = outliers[[col]].copy()
        print(f"--- Outliers found in column: {col} ---")
        print(f"Number of outliers: {len(outliers)}")
        print(outliers[[col]].head())
        print("\n")

if not outliers_data_cleaned:
    print("No significant outliers detected using the IQR method in numerical columns of df.")
else:
    print(f"Summary: Outliers found in {len(outliers_data_cleaned)} numerical columns of df.")

### Visualize Outliers in `df_cleaned` with Box Plots

Let's visualize any remaining outliers in `df_cleaned`.

In [ ]:
columns_with_outliers_cleaned = list(outliers_data_cleaned.keys())

if columns_with_outliers_cleaned:
    print(f"Generating box plots for {len(columns_with_outliers_cleaned)} columns with detected outliers in df.")

    n_cols = 3  # You can adjust this number
    n_rows = (len(columns_with_outliers_cleaned) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten() # Flatten the array of axes for easy iteration

    for i, col in enumerate(columns_with_outliers_cleaned):
        sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
        axes[i].set_title(f'Box Plot of {col} (df)')
        axes[i].set_ylabel('') # Hide y-label to avoid clutter

    # Hide any unused subplots
    for j in range(len(columns_with_outliers_cleaned), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()
else:
    print("No outliers were detected in numerical columns of df, so no box plots will be generated.")

In [ ]:
#df = df.drop('id', axis=1)
#print("The 'id' column has been dropped from the DataFrame.")
display(df.head())

##One_Hot Encoding

In [ ]:
categorical_cols = ['location_type', 'land_use', 'zoning_code']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("DataFrame after one-hot encoding:")
display(df.head())
print(f"\nShape of DataFrame after one-hot encoding: {df.shape}")

##Coorelation

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr()

print("Correlation Matrix (first 5 rows and columns):")
display(correlation_matrix.head())

### Visualize Correlation with Target Variable

To focus on the most relevant correlations for our prediction task, let's visualize the correlation of all features with the `target` variable.

In [ ]:
# Get correlations with the 'target' variable
target_correlations = correlation_matrix['target'].sort_values(ascending=False)

plt.figure(figsize=(10, 20))
sns.heatmap(target_correlations.to_frame(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation with Target Variable')
plt.show()

In [ ]:
# Drop the 'target' column before calculating correlation
df_features = df.drop(columns=['target'])

# Calculate the correlation matrix without the target variable
correlation_matrix_no_target = df_features.corr()

print("Correlation Matrix (without 'target' variable - first 5 rows and columns):")
display(correlation_matrix_no_target.head())

### Visualize Feature-to-Feature Correlation

To understand the relationships and potential multicollinearity among the features themselves, let's visualize the `correlation_matrix_no_target` using a heatmap.

In [ ]:
plt.figure(figsize=(25, 20))
sns.heatmap(correlation_matrix_no_target, annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Feature-to-Feature Correlation Matrix (Excluding Target)')
plt.show()

##Split Data

## Data Splitting

Now, let's split the dataset into training and validation sets. This is a crucial step to evaluate the performance of our model on unseen data. We'll use 80% of the data for training and 20% for validation.

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")

## Data Normalization

Now, let's normalize the features. Normalization helps in scaling the data to a standard range, which can prevent certain features from dominating the learning process and improve the performance of machine learning models. We will use `StandardScaler` to transform the features to have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data (X_train) and transform both X_train and X_val
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Convert the scaled arrays back to DataFrames, retaining column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)

print("Features have been scaled.")
display(X_train_scaled.head())

## Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize the Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully.")

### Model Evaluation

## Make Predictions on Test Data

In [ ]:
# Define the path to the test.csv file in Google Drive
test_csv_path = f"{drive_folder}/test.csv"

# Load the test dataset
df_test = pd.read_csv(test_csv_path)

print("Test data loaded successfully.")
display(df_test.head())

### Preprocessing Test Data

Apply the same preprocessing steps to the test data as were applied to the training data. This includes handling the 'id' column, one-hot encoding, and scaling numerical features. It's crucial to use the same `scaler` fitted on the training data to ensure consistency.

In [ ]:
# Keep the 'id' column for submission if needed, but drop it for prediction
test_ids = df_test['id']
X_test = df_test.drop('id', axis=1)

# Apply one-hot encoding to categorical columns using the same columns as training
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - this is critical to ensure test data has the same columns as training data
# Add missing columns (if any) that were in X_train but not in X_test_encoded
missing_cols = set(X_train.columns) - set(X_test_encoded.columns)
for c in missing_cols:
    X_test_encoded[c] = 0
# Remove extra columns (if any) that were in X_test_encoded but not in X_train
extra_cols = set(X_test_encoded.columns) - set(X_train.columns)
for c in extra_cols:
    X_test_encoded = X_test_encoded.drop(c, axis=1)

# Ensure the order of columns is the same as in X_train
X_test_final = X_test_encoded[X_train.columns]

# Scale the test features using the *same scaler* fitted on the training data
X_test_scaled = scaler.transform(X_test_final)

X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_final.columns)

print("Test data preprocessed and scaled successfully.")
display(X_test_scaled.head())

### Generate Predictions

In [ ]:
# Make predictions on the scaled test data
y_test_pred = model.predict(X_test_scaled)

# Display the first few predictions
print("Generated predictions for the test set:")
display(pd.DataFrame({'id': test_ids, 'predicted_target': y_test_pred}).head())

In [ ]:
# Make predictions on the training set
y_train_pred = model.predict(X_train_scaled)

# Evaluate training set predictions
mae_train = mean_absolute_error(y_train, y_train_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

print("Training Set Evaluation:")
print(f"  MAE: {mae_train:.2f}")
print(f"  MSE: {mse_train:.2f}")
print(f"  RMSE: {rmse_train:.2f}")
print(f"  R-squared: {r2_train:.2f}")

# Make predictions on the validation set
y_val_pred = model.predict(X_val_scaled)

# Evaluate validation set predictions
mae_val = mean_absolute_error(y_val, y_val_pred)
mse_val = mean_squared_error(y_val, y_val_pred)
rmse_val = np.sqrt(mse_val)
r2_val = r2_score(y_val, y_val_pred)

print("\nValidation Set Evaluation:")
print(f"  MAE: {mae_val:.2f}")
print(f"  MSE: {mse_val:.2f}")
print(f"  RMSE: {rmse_val:.2f}")
print(f"  R-squared: {r2_val:.2f}")

In [ ]:
# Create a submission DataFrame
submission_df = pd.DataFrame({'id': test_ids, 'target': y_test_pred})

# Save the submission DataFrame to a CSV file
submission_path = f"{drive_folder}/submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission file created successfully at: {submission_path}")
display(submission_df.head())

In [ ]:
# Create a submission DataFrame
submission_df = pd.DataFrame({'id': test_ids, 'target': y_test_pred})

# Save the submission DataFrame to a CSV file
submission_path = f"{drive_folder}/submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission file created successfully at: {submission_path}")
display(submission_df.head())

In [ ]:
submission_df.shape

## Re-scaling Features for Model Comparison

Since the polynomial features led to overfitting, we will revert to the original scaled features (without polynomial transformation) for a fair comparison of Linear, Ridge, and Lasso Regression models.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler_revert = StandardScaler()

# Fit the scaler on the original training data (X_train) and transform both X_train and X_val
# Assuming X_train and X_val still hold the one-hot encoded, non-polynomial features
X_train_scaled_orig = scaler_revert.fit_transform(X_train)
X_val_scaled_orig = scaler_revert.transform(X_val)

# Convert the scaled arrays back to DataFrames, retaining column names
X_train_scaled_orig = pd.DataFrame(X_train_scaled_orig, columns=X_train.columns)
X_val_scaled_orig = pd.DataFrame(X_val_scaled_orig, columns=X_val.columns)

print("Features have been re-scaled to their original non-polynomial state.")
display(X_train_scaled_orig.head())

## Model Comparison: Linear Regression, Ridge, and Lasso

Now, let's train and evaluate three different linear models: standard Linear Regression, Ridge Regression (with L2 regularization), and Lasso Regression (with L1 regularization). Regularization can help prevent overfitting by adding a penalty to the model's coefficients.

## Generate Submission with Ridge Regression Model

In [ ]:
# Load the test dataset (already done, but re-defining for clarity in this section)
test_csv_path = f"{drive_folder}/test.csv"
df_test = pd.read_csv(test_csv_path)

# Keep the 'id' column for submission
test_ids_for_submission = df_test['id']
X_test_raw = df_test.drop('id', axis=1)

# Apply one-hot encoding to categorical columns using the same columns as training
X_test_encoded_for_submission = pd.get_dummies(X_test_raw, columns=categorical_cols, drop_first=True)

# Align columns - this is critical to ensure test data has the same columns as training data (X_train)
missing_cols_test = set(X_train.columns) - set(X_test_encoded_for_submission.columns)
for c in missing_cols_test:
    X_test_encoded_for_submission[c] = 0
extra_cols_test = set(X_test_encoded_for_submission.columns) - set(X_train.columns)
for c in extra_cols_test:
    X_test_encoded_for_submission = X_test_encoded_for_submission.drop(c, axis=1)

# Ensure the order of columns is the same as in X_train
X_test_final_for_submission = X_test_encoded_for_submission[X_train.columns]

# Scale the test features using the *scaler_revert* fitted on the original training data
X_test_scaled_for_submission = scaler_revert.transform(X_test_final_for_submission)

X_test_scaled_for_submission = pd.DataFrame(X_test_scaled_for_submission, columns=X_test_final_for_submission.columns)

print("Test data preprocessed and scaled for Ridge Model successfully.")
display(X_test_scaled_for_submission.head())

In [ ]:
# Make predictions using the Ridge Regression model
y_test_pred_ridge = ridge_model.predict(X_test_scaled_for_submission)

# Create a submission DataFrame
submission1_df = pd.DataFrame({'id': test_ids_for_submission, 'target': y_test_pred_ridge})

# Save the submission DataFrame to a CSV file named submission1.csv
submission1_path = f"{drive_folder}/submission1.csv"
submission1_df.to_csv(submission1_path, index=False)

print(f"Submission file created successfully at: {submission1_path}")
display(submission1_df.head())

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# --- 1. Linear Regression ---
print("\n--- Linear Regression Model ---")
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train_scaled_orig, y_train)

y_train_pred_lr = lin_reg_model.predict(X_train_scaled_orig)
y_val_pred_lr = lin_reg_model.predict(X_val_scaled_orig)

mae_train_lr = mean_absolute_error(y_train, y_train_pred_lr)
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
r2_train_lr = r2_score(y_train, y_train_pred_lr)

mae_val_lr = mean_absolute_error(y_val, y_val_pred_lr)
rmse_val_lr = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))
r2_val_lr = r2_score(y_val, y_val_pred_lr)

print(f"Training MAE: {mae_train_lr:.2f}, RMSE: {rmse_train_lr:.2f}, R2: {r2_train_lr:.2f}")
print(f"Validation MAE: {mae_val_lr:.2f}, RMSE: {rmse_val_lr:.2f}, R2: {r2_val_lr:.2f}")

# --- 2. Ridge Regression ---
print("\n--- Ridge Regression Model ---")
# You can tune the alpha parameter. A higher alpha increases regularization strength.
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train_scaled_orig, y_train)

y_train_pred_ridge = ridge_model.predict(X_train_scaled_orig)
y_val_pred_ridge = ridge_model.predict(X_val_scaled_orig)

mae_train_ridge = mean_absolute_error(y_train, y_train_pred_ridge)
rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
r2_train_ridge = r2_score(y_train, y_train_pred_ridge)

mae_val_ridge = mean_absolute_error(y_val, y_val_pred_ridge)
rmse_val_ridge = np.sqrt(mean_squared_error(y_val, y_val_pred_ridge))
r2_val_ridge = r2_score(y_val, y_val_pred_ridge)

print(f"Training MAE: {mae_train_ridge:.2f}, RMSE: {rmse_train_ridge:.2f}, R2: {r2_train_ridge:.2f}")
print(f"Validation MAE: {mae_val_ridge:.2f}, RMSE: {rmse_val_ridge:.2f}, R2: {r2_val_ridge:.2f}")

# --- 3. Lasso Regression ---
print("\n--- Lasso Regression Model ---")
# You can tune the alpha parameter. A higher alpha increases regularization strength.
lasso_model = Lasso(alpha=0.1, random_state=42, max_iter=2000) # Increased max_iter for convergence
lasso_model.fit(X_train_scaled_orig, y_train)

y_train_pred_lasso = lasso_model.predict(X_train_scaled_orig)
y_val_pred_lasso = lasso_model.predict(X_val_scaled_orig)

mae_train_lasso = mean_absolute_error(y_train, y_train_pred_lasso)
rmse_train_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
r2_train_lasso = r2_score(y_train, y_train_pred_lasso)

mae_val_lasso = mean_absolute_error(y_val, y_val_pred_lasso)
rmse_val_lasso = np.sqrt(mean_squared_error(y_val, y_val_pred_lasso))
r2_val_lasso = r2_score(y_val, y_val_pred_lasso)

print(f"Training MAE: {mae_train_lasso:.2f}, RMSE: {rmse_train_lasso:.2f}, R2: {r2_train_lasso:.2f}")
print(f"Validation MAE: {mae_val_lasso:.2f}, RMSE: {rmse_val_lasso:.2f}, R2: {r2_val_lasso:.2f}")